In [ ]:
# Cell 1: Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 2: Configuration and Setup
# Set the random seed for reproducibility as specified in the question
RANDOM_SEED = 6740
np.random.seed(RANDOM_SEED)

# Define parameters for the two normal distributions
# f0 = N(mu0, sigma0^2)
mu0 = 0
sigma0 = 1.0 # Standard deviation for f0 (variance is 1)

# f1 = N(mu1, sigma1^2)
mu1 = 0
sigma1 = np.sqrt(1.3) # Standard deviation for f1 (variance is 1.3)

# Define the number of samples from each distribution
n0 = 100 # Number of samples from f0 (pre-change)
n1 = 50  # Number of samples from f1 (post-change)
n_total = n0 + n1 # Total number of samples

print(f"Random seed set to: {RANDOM_SEED}")
print(f"Distribution f0: Normal(mean={mu0}, variance={sigma0**2:.1f})")
print(f"Distribution f1: Normal(mean={mu1}, variance={sigma1**2:.1f})")
print(f"Total samples: {n_total} (first {n0} from f0, next {n1} from f1)")

In [ ]:
# Cell 3: Data Generation
# Generate samples from f0
data_f0 = np.random.normal(mu0, sigma0, n0)

# Generate samples from f1
data_f1 = np.random.normal(mu1, sigma1, n1)

# Concatenate the data to form the full sequence
x_sequence = np.concatenate((data_f0, data_f1))

print("\nData sequence generated.")
print(f"First 5 samples: {x_sequence[:5].round(2)}")
print(f"Samples around change point (98-102): {x_sequence[98:103].round(2)}")
print(f"Last 5 samples: {x_sequence[-5:].round(2)}")

In [ ]:
# Cell 4: Calculate Log-Likelihood Ratios (l_k)
# The log-likelihood ratio l_k = log(f1(x_k) / f0(x_k))
# For a Normal distribution N(mu, sigma^2), the PDF is:
# f(x; mu, sigma^2) = (1 / sqrt(2 * pi * sigma^2)) * exp(- (x - mu)^2 / (2 * sigma^2))
#
# log(f(x; mu, sigma^2)) = -0.5 * log(2 * pi) - 0.5 * log(sigma^2) - (x - mu)^2 / (2 * sigma^2)
#
# l_k = log(f1(x_k)) - log(f0(x_k))
# l_k = [-0.5 * log(2 * pi) - 0.5 * log(sigma1^2) - (x_k - mu1)^2 / (2 * sigma1^2)] - \
#       [-0.5 * log(2 * pi) - 0.5 * log(sigma0^2) - (x_k - mu0)^2 / (2 * sigma0^2)]
#
# Simplifying, the -0.5 * log(2 * pi) terms cancel out:
# l_k = 0.5 * log(sigma0^2) - 0.5 * log(sigma1^2) - (x_k - mu1)^2 / (2 * sigma1^2) + (x_k - mu0)^2 / (2 * sigma0^2)
# l_k = log(sigma0 / sigma1) - (x_k - mu1)^2 / (2 * sigma1^2) + (x_k - mu0)^2 / (2 * sigma0^2)

# Given mu0 = 0, mu1 = 0:
# l_k = log(sigma0 / sigma1) - x_k^2 / (2 * sigma1^2) + x_k^2 / (2 * sigma0^2)
# l_k = log(sigma0 / sigma1) + x_k^2 * (1 / (2 * sigma0^2) - 1 / (2 * sigma1^2))
# l_k = log(sigma0 / sigma1) + x_k^2 * ( (sigma1^2 - sigma0^2) / (2 * sigma0^2 * sigma1^2) )

# Substituting values:
# sigma0 = 1, sigma1 = sqrt(1.3) => sigma0^2 = 1, sigma1^2 = 1.3
# l_k = log(1 / sqrt(1.3)) + x_k^2 * ( (1.3 - 1) / (2 * 1 * 1.3) )
# l_k = -0.5 * log(1.3) + x_k^2 * ( 0.3 / 2.6 )
# l_k = -0.5 * log(1.3) + x_k^2 * ( 3 / 26 )

# Calculate the constant part:
constant_term = -0.5 * np.log(1.3) # Using natural log as np.log is natural log
coefficient_x_squared = (3 / 26)

log_likelihood_ratios = constant_term + coefficient_x_squared * (x_sequence**2)

print("\nLog-likelihood ratios calculated.")
print(f"Constant term in l_k: {constant_term:.4f}")
print(f"Coefficient for x_k^2 in l_k: {coefficient_x_squared:.4f}")
print(f"First 5 l_k values: {log_likelihood_ratios[:5].round(4)}")
print(f"l_k values around change point (98-102): {log_likelihood_ratios[98:103].round(4)}")
print(f"Last 5 l_k values: {log_likelihood_ratios[-5:].round(4)}")


In [ ]:
# Cell 5: Calculate CUSUM Statistics (S_k)
# S_k = max(0, S_{k-1} + l_k), with S_0 = 0

cusum_statistics = np.zeros(n_total + 1) # Initialize S_0 to S_n_total. S_0 is at index 0.

for k in range(n_total):
    # The current S_k is stored at index k+1 in the array
    # because S_0 is at index 0, S_1 at index 1, etc.
    cusum_statistics[k+1] = max(0, cusum_statistics[k] + log_likelihood_ratios[k])

print("\nCUSUM statistics calculation complete.")
print(f"S_0: {cusum_statistics[0]:.4f}")
print(f"S_{n0-1}: {cusum_statistics[n0-1]:.4f}")
print(f"S_{n0}: {cusum_statistics[n0]:.4f}") # This is S_k before processing x_k
print(f"S_{n0+1}: {cusum_statistics[n0+1]:.4f}") # This is S_k after processing x_{n0} (the first post-change observation)
print(f"S_{n_total}: {cusum_statistics[n_total]:.4f}")

In [ ]:
# Cell 6: Plotting the CUSUM Statistic
plt.figure(figsize=(12, 6))
plt.plot(range(n_total + 1), cusum_statistics, marker='o', linestyle='-', markersize=3, label='CUSUM Statistic $S_k$')

# Mark the true change point
plt.axvline(x=n0, color='r', linestyle='--', label=f'True Change Point (k={n0})')

plt.xlabel('Observation Index $k$')
plt.ylabel('CUSUM Statistic $S_k$')
plt.title('CUSUM Statistic for Change Point Detection (Seed: 6740)')
plt.grid(True)
plt.legend()
plt.tight_layout()

# Save the plot to a file
plot_filename = 'cusum_plot_q4b.png' # Changed filename to avoid conflict if you ran before
plt.savefig(plot_filename)
print(f"\nCUSUM plot saved as '{plot_filename}'")

plt.show()
